<html><head><meta content="text/html; charset=UTF-8" http-equiv="content-type"><style type="text/css">ol</style></head><body class="c5"><p class="c0 c4"><span class="c3"></span></p><p class="c2 title" id="h.rrbabt268i6e"><h1>CaImAn&rsquo;s Demo pipeline</h1></p><p class="c0"><span class="c3">This notebook will help to demonstrate the process of CaImAn and how it uses different functions to denoise, deconvolve and demix neurons from a two-photon Calcium Imaging dataset. The demo shows how to construct the `params`, `MotionCorrect` and `cnmf` objects and call the relevant functions. You can also run a large part of the pipeline with a single method (`cnmf.fit_file`). See inside for details.

Dataset couresy of Sue Ann Koay and David Tank (Princeton University)

This demo pertains to two photon data. For a complete analysis pipeline for one photon microendoscopic data see demo_pipeline_cnmfE.ipynb</span></p>
<p class="c0"><span class="c3">More information can be found in the companion paper. </span></p>
</html>

In [ ]:
import bokeh.plotting as bpl
import cv2
import glob
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import scipy.io
import tifffile

try:
    cv2.setNumThreads(0)
except():
    pass

try:
    if __IPYTHON__:
        # this is used for debugging purposes only. allows to reload classes
        # when changed
        get_ipython().magic('load_ext autoreload')
        get_ipython().magic('autoreload 2')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf as cnmf
from caiman.source_extraction.cnmf import params as params
from caiman.utils.utils import download_demo
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
bpl.output_notebook()

### Set up logger (optional)
You can log to a file using the filename parameter, or make the output more or less verbose by setting level to `logging.DEBUG`, `logging.INFO`, `logging.WARNING`, or `logging.ERROR`. A filename argument can also be passed to store the log file

In [ ]:
logging.basicConfig(format=
                          "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s",
                    # filename="/tmp/caiman.log",
                    level=logging.WARNING)

### Link single tiff to multipage tiff (optional)
You can chain tiff images from a single run or from multiple runs if the same ROI was imaged multiple times.

In [ ]:
chain_tiff = True

savedir = 'H:\\Analysis\\2P\\VG1GC6_H1M1\\12162022\\run12347\\'

if chain_tiff:
    sourcedir1 = 'D:\\Data\\2P\\VG1GC6_H1M1\\12162022\\TSeries-12162022-1355-001'
    sourcedir2 = 'D:\\Data\\2P\\VG1GC6_H1M1\\12162022\\TSeries-12162022-1355-002'
    sourcedir3 = 'D:\\Data\\2P\\VG1GC6_H1M1\\12162022\\TSeries-12162022-1355-003'
    sourcedir4 = 'D:\\Data\\2P\\VG1GC6_H1M1\\12162022\\TSeries-12162022-1355-004'
    sourcedir5 = 'D:\\Data\\2P\\VG1GC6_H1M1\\12162022\\TSeries-12162022-1355-007'
  #  sourcedir6 = 'D:\\Data\\2P\\VG1GC6_H1M0\\12082022\\TSeries-12082022-1355-006'

    fls = [glob.glob(os.path.join(sourcedir1,'*_Ch2_*.ome.tif')),
       glob.glob(os.path.join(sourcedir2,'*_Ch2_*.ome.tif')),
       glob.glob(os.path.join(sourcedir3,'*_Ch2_*.ome.tif')),
       glob.glob(os.path.join(sourcedir4,'*_Ch2_*.ome.tif')),
       glob.glob(os.path.join(sourcedir5,'*_Ch2_*.ome.tif'))]
   #    glob.glob(os.path.join(sourcedir6,'*_Ch2_*.ome.tif'))

    fls.sort()  # make sure your files are sorted alphanumerically

    m = cm.load_movie_chain(fls,outtype=np.ushort)

    m.save(os.path.join(savedir,'data.h5'),to32=False)

       10538 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10539 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10546 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10547 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10553 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10553 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10560 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10561 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10568 [tifffile.py:         log_warning():16549] 

       10794 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10801 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10802 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10809 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10809 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10816 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10816 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       10824 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       10824 [movies.py:                load():1515] [26

       11059 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11066 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11066 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11074 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11074 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11083 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11083 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11091 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11092 [movies.py:                load():1515] [26

       11327 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11328 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11335 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11335 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11342 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11343 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11350 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11350 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11357 [tifffile.py:         log_warning():16549] 

       11589 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11589 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11597 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11598 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       11605 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11605 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected

  9%|███████▎                                                                      | 170/1800 [00:01<00:12, 128.69it/s]       11614 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       11614 [movies.py:                load():1515] [26820] Your ti

...

 92%|███████████████████████████████████████████████████████████████████████      | 1662/1800 [00:12<00:01, 129.50it/s]       83463 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83463 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83469 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83471 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83478 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83478 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83485 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83486 [movies.py:                load():1515] [26820] Your tif

       83713 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83721 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83721 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83728 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83729 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83736 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83736 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83743 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83744 [movies.py:                load():1515] [26

       83972 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83979 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83979 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       83987 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83987 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected

 96%|██████████████████████████████████████████████████████████████████████████   | 1732/1800 [00:13<00:00, 131.05it/s]       83996 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       83997 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       84004 [tifffile.py:         log_w

       84233 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       84234 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       84241 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       84241 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       84249 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       84250 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       84257 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       84257 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       84264 [tifffile.py:         log_warning():16549] 

       84495 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       84495 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       84502 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       84502 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
       84510 [tifffile.py:         log_warning():16549] [26820] OME series: not an ome-tiff master file
       84510 [movies.py:                load():1515] [26820] Your tif file is saved a single pagefile. Performance will be affected
100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:15<00:00, 15.07s/it]
      109667 [timeseries.py:                save():278] [26820] No file saved


### Select file(s) to be processed
The `download_demo` function will download the specific file for you and return the complete path to the file which will be stored in your `caiman_data` directory. If you adapt this demo for your data make sure to pass the complete path to your file(s). Remember to pass the `fname` variable as a list.

In [ ]:
fnames = [savedir + 'data.h5']  # filename to be processed
#if fnames[0] in ['data.tif', 'demoMovie.tif']:
#    fnames = [download_demo(fnames[0])]

### Play the movie (optional)
Play the movie (optional). This will require loading the movie in memory which in general is not needed by the pipeline. Displaying the movie uses the OpenCV library. Press `q` to close the video panel.

In [ ]:
m_orig = cm.load_movie_chain(fnames)
display_movie = False
if display_movie:    
    ds_ratio = 1
    m_orig.resize(1, 1, ds_ratio).play(
        q_max=99.5, fr=7.5, magnification=1)

### Setup some parameters
We set some parameters that are relevant to the file, and then parameters for motion correction, processing with CNMF and component quality evaluation. Note that the dataset `Sue_2x_3000_40_-46.tif` has been spatially downsampled by a factor of 2 and has a lower than usual spatial resolution (2um/pixel). As a result several parameters (`gSig, strides, max_shifts, rf, stride_cnmf`) have lower values (halved compared to a dataset with spatial resolution 1um/pixel).

In [ ]:
# load params or save them if they do not exist

save_params = True

if save_params:    
   
    # dataset dependent parameters
    fr = 15                             # imaging rate in frames per second
    decay_time = 1.2                    # length of a typical transient in seconds
    stim_seq = (2000,2000,120,120,2000)     # stim intensity values
    stim_left_side = 1
    Ntrial = (20)                         
    trial_duration = 6                  # in s
    stim_delay = 2                      # in s
    Npixel_x = 1024
    Npixel_y = 1024
    objective ='16x'
    numerical_zoom = 0.8
    behavior_run_names = ['run1','run2','run3','run4','run7']
 
    # 16x zoom0.8 1024x1024
    # motion correction parameters
    strides = (80, 80)          # start a new patch for pw-rigid motion correction every x pixels
    overlaps = (40, 40)         # overlap between pathes (size of patch strides+overlaps)
    max_shifts = (16, 16)        # maximum allowed rigid shifts (in pixels)
    max_deviation_rigid = 8     # maximum shifts deviation allowed for patch with respect to rigid shifts
    pw_rigid = True             # flag for performing non-rigid motion correction

    # parameters for source extraction and deconvolution
    p = 1                       # order of the autoregressive system
    gnb = 2                     # number of global background components
    merge_thr = 0.85             # merging threshold, max correlation allowed
    rf = 20                     # half-size of the patches in pixels. e.g., if rf=25, patches are 50x50
    stride_cnmf = 8             # amount of overlap between the patches in pixels
    K = 8                       # number of components per patch
    gSig = [4, 4]               # expected half size of neurons in pixels
    method_init = 'greedy_roi'  # initialization method (if analyzing dendritic data using 'sparse_nmf')
    ssub = 1                    # spatial subsampling during initialization
    tsub = 1                    # temporal subsampling during intialization
   
    # parameters for component evaluation
    SNR_lowest = 1.5
    min_SNR = 2.5              # signal to noise ratio for accepting a component
    rval_lowest = 0.4
    rval_thr = 0.85              # space correlation threshold for accepting a component
    cnn_thr = 0.95              # threshold for CNN based classifier
    cnn_lowest = 0.2            # neurons with cnn probability lower than this value are rejected
    
    # save params to matlab
    scipy.io.savemat(savedir + 'params.mat', mdict={'fr': fr,
                                                    'decay_time': decay_time,
                                                    'stim_seq': stim_seq,
                                                    'stim_left_side': stim_left_side,
                                                    'Ntrial': Ntrial,
                                                    'trial_duration': trial_duration,
                                                    'stim_delay': stim_delay,
                                                    'Npixel_x': Npixel_x,
                                                    'Npixel_y': Npixel_y,
                                                    'objective': objective,
                                                    'numerical_zoom': numerical_zoom,
                                                    'behavior_run_names': behavior_run_names,
                                                    'strides': strides,
                                                    'overlaps': overlaps,
                                                    'max_shifts': max_shifts,
                                                    'max_deviation_rigid': max_deviation_rigid,
                                                    'pw_rigid': pw_rigid,
                                                    'p': p,
                                                    'gnb': gnb,
                                                    'merge_thr': merge_thr,
                                                    'rf': rf,
                                                    'stride_cnmf': stride_cnmf,
                                                    'K': K,
                                                    'gSig': gSig,
                                                    'method_init': method_init,
                                                    'ssub': ssub,
                                                    'tsub': tsub,
                                                    'min_SNR': min_SNR,
                                                    'SNR_lowest': SNR_lowest,
                                                    'rval_thr': rval_thr,
                                                    'rval_lowest': rval_lowest,
                                                    'cnn_thr': cnn_thr,
                                                    'cnn_lowest': cnn_lowest})
else:
    myparams = scipy.io.loadmat(savedir + 'params.mat',squeeze_me=True)
   
    fr = myparams['fr']                    
    decay_time = myparams['decay_time'] 
    stim_seq = myparams['stim_seq'] 
    stim_left_side = myparams['stim_left_side']
    Ntrial = myparams['Ntrial']                          
    trial_duration = myparams['trial_duration'] 
    stim_delay = myparams['stim_delay'] 
    Npixel_x = myparams['Npixel_x']
    Npixel_y = myparams['Npixel_y']
    objective =myparams['objective']
    numerical_zoom = myparams['numerical_zoom']
    behavior_run_names = myparams['behavior_run_names']
   
    strides = myparams['strides'] 
    overlaps = myparams['overlaps'] 
    max_shifts = myparams['max_shifts'] 
    max_deviation_rigid = myparams['max_deviation_rigid']
    pw_rigid = True

    # parameters for source extraction and deconvolution
    p = myparams['p'] 
    gnb = myparams['gnb'] 
    merge_thr = myparams['merge_thr'] 
    rf = myparams['rf'] 
    stride_cnmf = myparams['stride_cnmf'] 
    K = myparams['K'] 
    gSig = myparams['gSig'] 
    method_init = myparams['method_init'] 
    ssub = myparams['ssub'] 
    tsub = myparams['tsub'] 

    # parameters for component evaluation
    min_SNR = myparams['min_SNR'] 
    SNR_lowest = myparams['SNR_lowest']
    rval_thr = myparams['rval_thr'] 
    rval_lowest = myparams['rval_lowest']
    cnn_thr = myparams['cnn_thr'] 
    cnn_lowest = myparams['cnn_lowest'] 
  



### Create a parameters object
You can creating a parameters object by passing all the parameters as a single dictionary. Parameters not defined in the dictionary will assume their default values. The resulting `params` object is a collection of subdictionaries pertaining to the dataset to be analyzed `(params.data)`, motion correction `(params.motion)`, data pre-processing `(params.preprocess)`, initialization `(params.init)`, patch processing `(params.patch)`, spatial and temporal component `(params.spatial), (params.temporal)`, quality evaluation `(params.quality)` and online processing `(params.online)`

In [ ]:
opts_dict = {'fnames': fnames,
            'fr': fr,
            'decay_time': decay_time,
            'strides': strides,
            'overlaps': overlaps,
            'max_shifts': max_shifts,
            'max_deviation_rigid': max_deviation_rigid,
            'pw_rigid': pw_rigid,
            'p': p,
            'nb': gnb,
            'rf': rf,
            'gSig': gSig, 
            'K': K, 
            'stride': stride_cnmf,
            'method_init': method_init,
            'rolling_sum': True,
            'only_init': True,
            'ssub': ssub,
            'tsub': tsub,
            'merge_thr': merge_thr, 
            'min_SNR': min_SNR,
            'SNR_lowest': SNR_lowest,
            'rval_thr': rval_thr,
            'rval_lowest': rval_lowest,
            'use_cnn': True,
            'min_cnn_thr': cnn_thr,
            'cnn_lowest': cnn_lowest}

opts = params.CNMFParams(params_dict=opts_dict)

### Setup a cluster
To enable parallel processing a (local) cluster needs to be set up. This is done with a cell below. The variable `backend` determines the type of cluster used. The default value `'local'` uses the multiprocessing package. The `ipyparallel` option is also available. More information on these choices can be found [here](https://github.com/flatironinstitute/CaImAn/blob/master/CLUSTER.md). The resulting variable `dview` expresses the cluster option. If you use `dview=dview` in the downstream analysis then parallel processing will be used. If you use `dview=None` then no parallel processing will be employed.

In [ ]:
#%% start a cluster for parallel processing (if a cluster already exists it will be closed and a new session will be opened)
if 'dview' in locals():
    cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
    backend='local', n_processes=None, single_thread=False)

## Motion Correction
First we create a motion correction object with the parameters specified. Note that the file is not loaded in memory

In [ ]:
# first we create a motion correction object with the parameters specified
mc = MotionCorrect(fnames, dview=dview, **opts.get_group('motion'))
# note that the file is not loaded in memory

Now perform motion correction. From the movie above we see that the dateset exhibits non-uniform motion. We will perform piecewise rigid motion correction using the NoRMCorre algorithm. This has already been selected by setting `pw_rigid=True` when defining the parameters object.

In [ ]:
%%capture
#%% Run piecewise-rigid motion correction using NoRMCorre
mc.motion_correct(save_movie=True)
m_els = cm.load(mc.fname_tot_els)
border_to_0 = 0 if mc.border_nan is 'copy' else mc.border_to_0 
    # maximum shift to be used for trimming against NaNs

Inspect the results by comparing the original movie. A more detailed presentation of the motion correction method can be found in the [demo motion correction](./demo_motion_correction.ipynb) notebook.

In [ ]:
#%% plot rigid shifts
plt.close()
plt.figure(figsize = (20,10))
plt.plot(mc.shifts_rig)
plt.legend(['x shifts','y shifts'])
plt.xlabel('frames')
plt.ylabel('pixels')

In [ ]:
#%% compare with original movie
mean_map_raw = np.mean(m_orig,0)
mean_map_motion_corrected = np.mean(m_els,0)
corr_map_raw = m_orig.local_correlations(eight_neighbours=True, swap_dim=False)
corr_map_motion_corrected = m_els.local_correlations(eight_neighbours=True, swap_dim=False)
plt.figure(figsize = (20,20))
plt.subplot(2,2,1); plt.imshow(mean_map_raw)
plt.subplot(2,2,2); plt.imshow(mean_map_motion_corrected)
plt.subplot(2,2,3); plt.imshow(corr_map_raw)
plt.subplot(2,2,4); plt.imshow(corr_map_motion_corrected)

display_movie = False
if display_movie:
    m_orig = cm.load_movie_chain(fnames)
    ds_ratio = 1
    cm.concatenate([m_orig.resize(1, 1, ds_ratio) - mc.min_mov*mc.nonneg_movie,
                    m_els.resize(1, 1, ds_ratio)], 
                   axis=2).play(fr=30, gain=1, magnification=1, offset=0)  # press q to exit
    


## 

In [ ]:
# save motion corrected movie
m_els.save(savedir + 'data_motion_corrected.tif')

## Memory mapping 

The cell below memory maps the file in order `'C'` and then loads the new memory mapped file. The saved files from motion correction are memory mapped files stored in `'F'` order. Their paths are stored in `mc.mmap_file`.

In [ ]:
#%% MEMORY MAPPING
# memory map the file in order 'C'
fname_new = cm.save_memmap(mc.mmap_file, base_name='memmap_', order='C',
                           border_to_0=border_to_0, dview=dview) # exclude borders

# now load the file
Yr, dims, T = cm.load_memmap(fname_new)
images = np.reshape(Yr.T, [T] + list(dims), order='F') 
    #load frames in python format (T x X x Y)

Now restart the cluster to clean up memory

In [ ]:
#%% restart cluster to clean up memory
cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
    backend='local', n_processes=None, single_thread=False)

## Run CNMF on patches in parallel

- The FOV is split is different overlapping patches that are subsequently processed in parallel by the CNMF algorithm.
- The results from all the patches are merged with special attention to idendtified components on the border.
- The results are then refined by additional CNMF iterations.

In [ ]:
%%capture
#%% RUN CNMF ON PATCHES
# First extract spatial and temporal components on patches and combine them
# for this step deconvolution is turned off (p=0). If you want to have
# deconvolution within each patch change params.patch['p_patch'] to a
# nonzero value
cnm = cnmf.CNMF(n_processes, params=opts, dview=dview)
cnm = cnm.fit(images)

## Run the entire pipeline up to this point with one command
It is possible to run the combined steps of motion correction, memory mapping, and cnmf fitting in one step as shown below. The command is commented out since the analysis has already been performed. It is recommended that you familiriaze yourself with the various steps and the results of the various steps before using it.

In [ ]:
#cnm1 = cnmf.CNMF(n_processes, params=opts, dview=dview)
#cnm1.fit_file(motion_correct=True)

### Inspecting the results
Briefly inspect the results by plotting contours of identified components against correlation image.
The results of the algorithm are stored in the object `cnm.estimates`. More information can be found in the definition of the `estimates` object and in the [wiki](https://github.com/flatironinstitute/CaImAn/wiki/Interpreting-Results).

In [ ]:
#%% plot contours of found components
Cn = cm.local_correlations(images.transpose(1,2,0))
Cn[np.isnan(Cn)] = 0
cnm.estimates.plot_contours_nb(img=Cn)

## Re-run (seeded) CNMF  on the full Field of View  
You can re-run the CNMF algorithm seeded on just the selected components from the previous step. Be careful, because components rejected on the previous step will not be recovered here.

In [ ]:
%%capture
#%% RE-RUN seeded CNMF on accepted patches to refine and perform deconvolution 
cnm2 = cnm.refit(images, dview=dview)

## Component Evaluation

The processing in patches creates several spurious components. These are filtered out by evaluating each component using three different criteria:

- the shape of each component must be correlated with the data at the corresponding location within the FOV
- a minimum peak SNR is required over the length of a transient
- each shape passes a CNN based classifier

In [ ]:
#%% COMPONENT EVALUATION
# the components are evaluated in three ways:
#   a) the shape of each component must be correlated with the data
#   b) a minimum peak SNR is required over the length of a transient
#   c) each shape passes a CNN based classifier

cnm2.estimates.evaluate_components(images, cnm2.params, dview=dview)

Plot contours of selected and rejected components

In [ ]:
#%% PLOT COMPONENTS
cnm2.estimates.plot_contours_nb(img=Cn, idx=cnm2.estimates.idx_components)

View traces of accepted and rejected components. Note that if you get data rate error you can start Jupyter notebooks using:
'jupyter notebook --NotebookApp.iopub_data_rate_limit=1.0e10'

In [ ]:
# accepted components
cnm2.estimates.nb_view_components(img=Cn, idx=cnm2.estimates.idx_components[1:1000])

In [ ]:
# rejected components
if len(cnm2.estimates.idx_components_bad) > 0:
    cnm2.estimates.nb_view_components(img=Cn, idx=cnm2.estimates.idx_components_bad[1:2])
else:
    print("No components were rejected.")

### Extract DF/F values

In [ ]:
#%% Extract DF/F values
cnm2.estimates.detrend_df_f(quantileMin=15, frames_window=1800)

### Select only high quality components

In [ ]:
cnm2.estimates.select_components(use_object=True)

## Display final results

In [ ]:
cnm2.estimates.nb_view_components(img=Cn, denoised_color='red',idx=range(1,500))
print('you may need to change the data rate to generate this one: use jupyter notebook --NotebookApp.iopub_data_rate_limit=1.0e10 before opening jupyter notebook')

## Deconvolved dff

In [ ]:
cnm2.estimates.deconvolve(params=opts,dview=None, dff_flag=True)

### Save to Matlab

In [ ]:
# save results for matlab
scipy.io.savemat(savedir + 'results_caiman.mat', mdict={'mean_map_raw': mean_map_raw,
                                                 'mean_map_motion_corrected': mean_map_motion_corrected,
                                                 'corr_map_raw': corr_map_raw,
                                                 'corr_map_motion_corrected': corr_map_motion_corrected,
                                                 'spatial_components': cnm2.estimates.A,
                                                 'temporal_components': cnm2.estimates.C,
                                                 'background_spatial_component': cnm2.estimates.b,
                                                 'background_temporal_component': cnm2.estimates.f,
                                                 'df_wo_bckgrnd': cnm2.estimates.F_dff,
                                                 'deconv_spk': cnm2.estimates.S_dff})


### View movie with the results
We can inspect the denoised results by reconstructing the movie and playing alongside the original data and the resulting (amplified) residual movie. The denoised movie is saved, as well as a separate movie of the motion corrected data.

In [ ]:
 cnm2.estimates.play_movie(images, q_max=99.9, gain_res=1,
                                  frame_range=slice(0, 1000),
                                  magnification=0.5,
                                  bpx=border_to_0,
                                  display=True,
                                  include_bck=True)

# reconstruct denoised movie
# denoised = cm.movie(cnm2.estimates.A.dot(cnm2.estimates.C) + \
#                    cnm2.estimates.b.dot(cnm2.estimates.f)).reshape(dims + (-1,), order='F').transpose([2, 0, 1])
# save motion corrected movie
# denoised.save(savedir + 'denoised_movie.tif')

### Stop cluster and clean up log files

In [ ]:
#%% STOP CLUSTER and clean up log files
cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)